# TP API NAKALA : Mise en place d'un dépôt par lot

## 1-Objectif du TP

Ce TP fait suite au notebook sur la [présentation de l'API de NAKALA](./presentation-api.ipynb). Il a pour objectif la mise en place d'une chaîne de dépôt par lot. Pour cela, nous allons partir d'un ensemble de fichiers images stockés dans le répertoire `./simple-dataset/img` et d'un fichier `./simple-dataset/dataset.csv` au [format CSV](https://fr.wikipedia.org/wiki/Comma-separated_values) regroupant les informations nécessaires à l'import de ces fichiers dans NAKALA.

## 2-Structure du fichier CSV

Nous avons structuré les informations nécessaires à l'import des fichiers de la manière suivante :

- chaque ligne correspond à une future donnée dans NAKALA et se compose des colonnes suivantes :
    - `file` : le nom des fichiers associés à la donnée séparés par des points-virgules
    - `status` : le statut de la donnée. Par précaution, nous allons uniquement créer des données avec le status `pending` sachant qu'en cas d'erreur, une donnnée `published` ne peut être supprimé que sur demande spécifique à l'équipe d'HUMA-NUM.
    - `type` : le type de la donnée. Nous avons renseigné directement une URL du référentiel de NAKALA
    - `title` : le titre de la donnée.
    - `author` : la liste des auteurs sous la forme "Nom,Prénom" séparés par des points-virgules.
    - `date` : la date de création de l'image au format AAAA, AAAA-MM ou AAAA-MM-JJ.
    - `licence` : la licence associée à la donnée. Nous avons renseigné directement une valeur du référentiel de NAKALA.
    - `description` : une courte description.
    - `keywords`: une liste de mots-clés.
    - `rights`: la liste des droits associés à la donnée sous la forme "id,role" séparés par des points-virgules.

On pourrait bien sûr imaginer une structure différente !

Voici une vue de ce que contient le fichier CSV :

In [1]:
import pandas as pd

df = pd.read_csv('simple-dataset/dataset.csv', sep=",")
# shows top 10 rows
df.head(10)
# do something

,file,status,type,title,author,date,licence,description,keywords,rights
0,kingfisher.jpg,pending,http://purl.org/coar/resource_type/c_c513,Martin-pêcheur,"Dupont,Jean",2000-10-21,CC-BY-4.0,Une photo de martin-pêcheur,oiseau;martin-pêcheur,"de0f2a9b-a198-48a4-8074-db5120187a16,ROLE_READER"
1,owl-1.jpg;owl-2.jpg,pending,http://purl.org/coar/resource_type/c_c513,Chouette,"Martin,Pierre;Dupont,Jean",2021,CC-BY-4.0,Deux photos de chouette,oiseau;chouette,"e0dfb4c6-a4f9-4db7-a0a6-d7c41be42be9,ROLE_EDIT..."
2,peafowl.jpg,pending,http://purl.org/coar/resource_type/c_c513,Paon,"Martin,Pierre",1999-03,CC-BY-4.0,Une photo de paon,oiseau;paon,NaN


## 3-Script d'import par lot : mise en place étape par étape

Nous allons mettre en place un script Python qui va :

- parcourir le fichier CSV et récupérer pour chaque ligne les informations nécessaires à la création de la donnée dans NAKALA
- faire une première série de requêtes pour uploader le ou les fichiers associés à une donnée et récupérer leurs identifiants
- construire le contenu en JSON nécessaire pour la création de la donnée à partir des informations récupérées depuis le fichier CSV et l'upload des fichiers
- faire la requête de création de la donnée
- afficher au fur et à mesure les messages de l'API
- logger dans un fichier CSV de sortie les retours de l'API avec notemment l'identifiant attribuer à chaque nouvelle donnée, l'identifiant des fichiers et les éventuels messages d'erreur de l'API.

### 3.1-Parcours du fichier CSV et récupération des métadonnées

Le premier objectif de notre script est de parcourir le contenu du fichier CSV pour récupérer le chemin des fichiers et les métadonnées nécessaires à la création de chaque donnée. Nous allons stocker dans des variables distinctes les valeurs des colonnes de notre fichier CSV (ex: `filenames`, `status`, `title`...). Le principal point de vigilence à cette étape est que certaines colonnes de notre fichier CSV contiennent plusieurs informations spérarées par un ou plusieurs caractères séparateurs. La colonne `filenames` peut par exemple contenir plusieurs noms de fichier séparés par un point-virgule.

In [2]:
import csv

# 1. Lecture du fichier CSV
with open('simple-dataset/dataset.csv', newline='') as f:
    reader = csv.reader(f)
    dataset = list(reader)
dataset.pop(0) # Suppression des titres des colonnes

# 2. Parcours des différentes lignes du fichier
for num, data in enumerate(dataset):
    
    # 2.1. Récupération des infos disponibles sur la donnée à créer
    filenames = data[0].split(';') # On distingue bien les valeurs séparées par des points-virgules
    status = data[1]
    datatype = data[2]
    title = data[3]
    authors = data[4].split(';')
    date = data[5]
    license = data[6]
    description = data[7]
    keywords = data[8].split(';')
    datarights = data[9].split(';')
    
    print('CREATION DE LA DONNEE ' + str(num) + ' : ' + title)

CREATION DE LA DONNEE 0 : Martin-pêcheur
CREATION DE LA DONNEE 1 : Chouette
CREATION DE LA DONNEE 2 : Paon


### 3.2-Upload des fichiers de chaque donnée

Maintenant que nous avons récupérer dans le fichier CSV les informations nécessaires à la création des données, nous allons commencer par uploader les fichiers de chaque donnée. À cette étape, il est important de stocker les informations retournées par l'API après chaque upload. Ces informations seront réutilisées ensuite lors des requêtes de création des données. Il faut aussi s'assurer que tous les fichiers d'une même donnée ont bien été envoyés.

In [3]:
import csv, requests, json

# Endpoint API & Clef d'API
apiUrl = 'https://apitest.nakala.fr'
apiKey = 'aae99aba-476e-4ff2-2886-0aaf1bfa6fd2'

# 1. Lecture du fichier CSV
with open('simple-dataset/dataset.csv', newline='') as f:
    reader = csv.reader(f)
    dataset = list(reader)
dataset.pop(0)

# 2. Parcours des différentes lignes du fichier
for num, data in enumerate(dataset):
    
    # 2.1. Récupération des infos disponibles sur la donnée à créer
    filenames = data[0].split(';')
    status = data[1]
    datatype = data[2]
    title = data[3]
    authors = data[4].split(';')
    date = data[5]
    license = data[6]
    description = data[7]
    keywords = data[8].split(';')
    datarights = data[9].split(';')
    
    print('CREATION DE LA DONNEE ' + str(num) + " : " + title)
    
    # 2.2. Envoi des fichiers à l'API
    files = [] # variable pour stocker les informations retournées en JSON par l'API à chaque upload.
    for filename in filenames: # on parcours l'ensemble des fichiers d'une donnée
        print('Envoi du fichier ' + filename + '...') # on affiche un message pour le suivi de l'upload
        # écriture de la requête à l'API (ne contient pas de body en JSON, mais un fichier et un clef d'API)
        payload={}
        postfiles=[('file',(filename,open('./simple-dataset/img/' + filename, 'rb'),'image/jpeg'))]
        headers = {'X-API-KEY': apiKey }
        # appel à l'API pour uploader le fichier
        response = requests.request('POST', apiUrl + '/datas/uploads', headers=headers, data=payload, files=postfiles)
        # affichage du code de réponse de l'API
        print(response.status_code)
        # si l'upload s'est bien passé, on stocke les informations retournés par l'API dans la variable 'files'
        if ( 201 == response.status_code ):
            files.append(json.loads(response.text))        
    
    # 2.3. on vérifie que tous les fichiers d'une même donnée ont bien été uploadés
    if (len(filenames) == len(files)): # le nombre de fichiers renseignés dans le fichier CSV correspond bien au nombre de fichiers stockés dans la variable files après appel à l'API
        print ("Tous les fichiers de la donnée " + str(num) + " ont bien été uploadés !")
    else:
        print ("Certains fichiers de la donnée " + str(num) + " n'ont pas pu être envoyés !")

CREATION DE LA DONNEE 0 : Martin-pêcheur
Envoi du fichier kingfisher.jpg...
201
Tous les fichiers de la donnée 0 ont bien été uploadés !
CREATION DE LA DONNEE 1 : Chouette
Envoi du fichier owl-1.jpg...
201
Envoi du fichier owl-2.jpg...
201
Tous les fichiers de la donnée 1 ont bien été uploadés !
CREATION DE LA DONNEE 2 : Paon
Envoi du fichier peafowl.jpg...
201
Tous les fichiers de la donnée 2 ont bien été uploadés !


### 3.4-Construction du contenu en JSON pour la création de chaque donnée

Maintenant que chaque fichier d'une donnée a été uploadé et que nous avons récupéré les informations associées à ces uploads, nous allons pouvoir passer à la création à proprement parler des données via l'API. Pour cela, nous allons devoir construire à partir des métadonnées récupérées dans le fichier CSV, le contenu en JSON qui sera envoyé à l'API. Pour le moment, nous allons uniquement afficher ce contenu une fois que nous l'aurons construit pour s'assurer qu'il semble bien formé.

In [4]:
import csv, requests, json, time

# Endpoint API & Clef d'API
apiUrl = 'https://apitest.nakala.fr'
apiKey = 'aae99aba-476e-4ff2-2886-0aaf1bfa6fd2'

# 1. Lecture du fichier CSV
with open('simple-dataset/dataset.csv', newline='') as f:
    reader = csv.reader(f)
    dataset = list(reader)
dataset.pop(0)

# 2. Parcours des différentes lignes du fichier
for num, data in enumerate(dataset):
    
    # 2.1. Récupération des infos disponibles sur la donnée à créer
    filenames = data[0].split(';')
    status = data[1]
    datatype = data[2]
    title = data[3]
    authors = data[4].split(';')
    date = data[5]
    license = data[6]
    description = data[7]
    keywords = data[8].split(';')
    datarights = data[9].split(';')
    
    print('CREATION DE LA DONNEE ' + str(num) + " : " + title)
    
    # 2.2. Envoi des fichiers à l'API
    files = []
    for filename in filenames:
        print('Envoi du fichier ' + filename + '...')
        payload={}
        postfiles=[('file',(filename,open('./simple-dataset/img/' + filename, 'rb'),'image/jpeg'))]
        headers = {'X-API-KEY': apiKey }
        response = requests.request('POST', apiUrl + '/datas/uploads', headers=headers, data=payload, files=postfiles)
        print(response.status_code)
        if ( 201 == response.status_code ):
            # Attention ! Avant de stocker les informations retournées par l'API sur le fichier, on y ajoute une date d'embargo
            file = json.loads(response.text)
            file["embargoed"] = time.strftime("%Y-%m-%d") # On renseigne la date du jour
            files.append(file)
    
    # 2.3. on vérifie que tous les fichiers sont bien passés avant de créer la donnée
    if (len(files) != len(filenames)):
        print ("Certains fichiers n'ont pas pu être envoyés, on passe à la donnée suivante...")
        continue
    
    # 2.4. Reconstruction des métadonnées
    metas = [] # On stocke dans cette variable l'ensemble des métadonnées dans le format attendu
    
    # la métadonnée type (obligatoire)
    metaType = {
        "value": datatype, # On insère ici le contenu de la colonne "datatype" (un seul type par donnnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#anyURI", # On indique ici que la valeur renseignée est de type URI
        "propertyUri": "http://nakala.fr/terms#type" # On indique ici le champ "type" issu du vocabulaire NAKALA
    }
    metas.append(metaType) # Ajout de la métadonnée dans le tableau "metas"
    
    # la métadonnée titre (obligatoire)
    metaTitle = {
        "value": title, # On insère ici le contenu de la colonne "title" (un seul titre par donnnée)
        "lang": "fr", # On indique ici la langue du titre (cf. ISO-639-1 ou ISO-639-3 pour les langues moins courantes)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string", # On indique ici que la valeur renseignée est une chaîne de caractères
        "propertyUri": "http://nakala.fr/terms#title" # On indique ici le champ "title" issu du vocabulaire de NAKALA
    }
    metas.append(metaTitle)
    
    # les métadonnées auteurs (obligatoire pour une donnée publiée)
    for author in authors: # La colonne "authors" peut comporter plusieurs valeurs qu'on parcours une à une
        # Pour chaque valeur, on sépare le nom et le prénom pour construire la métadonnée nakala:creator
        surnameGivenname = author.split(',')
        metaAuthor = {
            "value": { # La valeur de cette métadonnée n'est pas une simple chaîne de caractères, mais un objet composé d'au minimum deux propriétées (givenname et surname)
                "givenname": surnameGivenname[1], # On insère ici le prénom
                "surname": surnameGivenname[0] # On insère ici le nom de famille
            },
            "propertyUri": "http://nakala.fr/terms#creator" # On indique ici le champ "creator" issu du vocabulaire de NAKALA
        }
        metas.append(metaAuthor)
        
    # la métadonnée date de création (obligatoire pour une donnée publiée)
    metaCreated = {
        "value": date, # On insère ici le contenu de la colonne "date" (une seule date de création par donnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#created"
    }
    metas.append(metaCreated)
    
    # la métadonnée licence (obligatoire pour une donnée publiée)
    metaLicense = {
        "value": license, # On insère ici le contenu de la colonne "license" (une seule license par donnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#license"
    }
    metas.append(metaLicense)
    
    # la métadonnée description (facultative)
    metaDescription = {
        "value": description, # On insère ici le contenu de la colonne "description" (une seule description par donnée)
        "lang": "fr",
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://purl.org/dc/terms/description" # Notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
    }
    metas.append(metaDescription)
    
    # les métadonnées mots-clés (facultatives)
    for keyword in keywords: # On parcours les valeurs de la colonne "keywords"
        metaKeyword = {
            "value": keyword, # On insère ici la valeur d'un mot-clé
            "lang": "fr",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/subject" # Notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaKeyword)
    
    # 2.5. Reconstruction des droits
    rights = [] # On stocke les droits d'une donnée dans un tableau
    for dataright in datarights: # On parcours les valeurs de la colonne "rights"
        datarightSplit = dataright.split(',') # Chaque valeur est composée d'un id et d'un rôle séparé par une virgule. On sépare ces valeurs
        if len(datarightSplit) == 2: # On vérifie qu'on a bien deux valeurs uniquement
            right = { # On reconstruit l'objet "right" composé des propriétés "id" et "role"
                "id": datarightSplit[0],
                "role": datarightSplit[1]
            }
            rights.append(right) # On ajout l'objet "right" dans le tableau "rights"
    
    # 3. Affichage du contenu qui sera envoyé à l'API de NAKALA pour créer la donnée
    postdata = {
        "status" : status, # on insère ici le status de la donnée
        "files" : files, # on insère ici l'ensemble des fichiers à associer à la donnée
        "metas" : metas, # on insère ici les métadonnées à proprement parlé
        "rights": rights # on insère ici les droits
    }
    print('Voici le contenu de la requête de création de la donnée ' + str(num) + " : ")
    print(json.dumps(postdata, indent=4, ensure_ascii=False))

CREATION DE LA DONNEE 0 : Martin-pêcheur
Envoi du fichier kingfisher.jpg...
201
Voici le contenu de la requête de création de la donnée 0 : 
{
    "status": "pending",
    "files": [
        {
            "name": "kingfisher.jpg",
            "sha1": "f4ca1456fa44f99a11be0b58cacd11fe86162e2e",
            "embargoed": "2021-12-17"
        }
    ],
    "metas": [
        {
            "value": "http://purl.org/coar/resource_type/c_c513",
            "typeUri": "http://www.w3.org/2001/XMLSchema#anyURI",
            "propertyUri": "http://nakala.fr/terms#type"
        },
        {
            "value": "Martin-pêcheur",
            "lang": "fr",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://nakala.fr/terms#title"
        },
        {
            "value": {
                "givenname": "Jean",
                "surname": "Dupont"
            },
            "propertyUri": "http://nakala.fr/terms#creator"
        },
        {
            "

### 3.5-Envoi des données à l'API et création d'un fichier de log

Les fichiers ont bien été uploadés. Le contenu en JSON de nos prochaines requêtes à l'API pour créer les donnnées semble correct. Nous allons maintenant pouvoir envoyer nos requêtes à l'API. Pour garder une trace de l'import par lot, nous allons en plus d'afficher quelques messages au fur et à mesure de l'exécution de notre script écrire les informations les plus importantes dans un fichier de sortie `output.csv`. Chaque ligne de ce fichier CSV sera composé des colonnes suivantes :

- `identifier` : l'identifiant associé à la donnée après sa création (ex: 10.34847/nkl.722a07wu)
- `files` : la listes des fichiers de la donnée avec l'identifiant associé à chaque fichier après l'upload. On utilisera la virgule et le point virgule comme séparateurs. (ex: owl-1.jpg,79e190850ce0eb840eb853671749f0789fb91217;owl-2.jpg,23c741d8cffe6c06ec9c9008597c74571dc348e0)
- `title` : le titre de la donnée (ex: Paon)
- `status` : le résultat de l'import. On prévoit simplement un label "OK" si tout s'est bien passé et un label "ERROR" en cas de problème.
- `response` : le contenu de la dernière réponse obtenue de l'API

In [5]:
import csv, requests, json, time

# Endpoint API & Clef d'API
apiUrl = 'https://apitest.nakala.fr'
apiKey = 'aae99aba-476e-4ff2-2886-0aaf1bfa6fd2'

# 1.1 Lecture du fichier CSV
with open('simple-dataset/dataset.csv', newline='') as f:
    reader = csv.reader(f)
    dataset = list(reader)
dataset.pop(0)

# 1.2 Préparation d'un fichier de sortie
output = open('output.csv', 'w') # ouverture d'un fichier en mode écriture
outputWriter = csv.writer(output) # création d'un objet pour écrire dans ce fichier
header = ['identifier', 'files', 'title', 'status', 'response'] # nom des colonnes à insérer dans ce fichier
outputWriter.writerow(header) # écriture du nom des colonnes dans ce fichier

# 2. Parcours des différentes lignes du fichier
for num, data in enumerate(dataset):
    
    # 2.1. Récupération des infos disponibles sur la donnée à créer
    filenames = data[0].split(';')
    status = data[1]
    datatype = data[2]
    title = data[3]
    authors = data[4].split(';')
    date = data[5]
    license = data[6]
    description = data[7]
    keywords = data[8].split(';')
    datarights = data[9].split(';')
    
    outputData = ['','',title,'','']; # variable où seront stockées les informations à écrire dans le fichier de sortie
    
    print('CREATION DE LA DONNEE ' + str(num) + " : " + title)
    
    # 2.2. Envoi des fichiers à l'API
    files = []
    outputFiles = [] # variable pour stocker les informations à écrire dans le fichier de sortie concernant les fichiers uploadés
    for filename in filenames:
        goToNextData = False # variable pour savoir s'il faut passer à la donnée suivante en cas d'erreur lors d'un upload
        print('Envoi du fichier ' + filename + '...')
        payload={}
        postfiles=[('file',(filename,open('./simple-dataset/img/' + filename, 'rb'),'image/jpeg'))]
        headers = {'X-API-KEY': apiKey }
        response = requests.request('POST', apiUrl + '/datas/uploads', headers=headers, data=payload, files=postfiles)
        if ( 201 == response.status_code ):
            file = json.loads(response.text)
            file["embargoed"] = time.strftime("%Y-%m-%d") # On renseigne la date du jour
            files.append(file)
            # Ajout du nom du fichier et du sha1 aux informations à écrire dans le fichier de sortie
            outputFiles.append(filename + ',' + file["sha1"])
        else:
            # Une erreur s'est produite avec un upload
            outputFiles.append(filename)
            outputData[1] = ';'.join(outputFiles)
            outputData[3] = 'ERROR'
            outputData[4] = response.text
            outputWriter.writerow(outputData)
            print ("Certains fichiers n'ont pas pu être envoyés, on passe à la donnée suivante...")
            goToNextData = True # on indique dans cette variable qu'il faut passer à la donnée suivante
            break # on stop l'upload des autres fichiers de la donnée
    if goToNextData: continue # on passe à la donnée suivante s'il y a eu une erreur
    
    # 2.3. On garde une trace pour le fichier de sortie de la liste des fichiers uploadés
    outputData[1] = ';'.join(outputFiles)
    
    # 2.4. Reconstruction des métadonnées
    metas = []
    
    # la métadonnée type (obligatoire)
    metaType = {
        "value": datatype,
        "typeUri": "http://www.w3.org/2001/XMLSchema#anyURI",
        "propertyUri": "http://nakala.fr/terms#type"
    }
    metas.append(metaType)
    
    # la métadonnée titre (obligatoire)
    metaTitle = {
        "value": title,
        "lang": "fr",
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#title"
    }
    metas.append(metaTitle)
    
    # les métadonnées auteurs (obligatoire pour une donnée publiée)
    for author in authors:
        surnameGivenname = author.split(',')
        metaAuthor = {
            "value": {
                "givenname": surnameGivenname[1],
                "surname": surnameGivenname[0]
            },
            "propertyUri": "http://nakala.fr/terms#creator"
        }
        metas.append(metaAuthor)
        
    # la métadonnée date de création (obligatoire pour une donnée publiée)
    metaCreated = {
        "value": date,
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#created"
    }
    metas.append(metaCreated)
    
    # la métadonnée licence (obligatoire pour une donnée publiée)
    metaLicense = {
        "value": license,
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#license"
    }
    metas.append(metaLicense)
    
    # la métadonnée description (facultative)
    metaDescription = {
        "value": description,
        "lang": "fr",
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://purl.org/dc/terms/description"
    }
    metas.append(metaDescription)
    
    # les métadonnées mots-clés (facultatives)
    for keyword in keywords:
        metaKeyword = {
            "value": keyword,
            "lang": "fr",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/subject"
        }
        metas.append(metaKeyword)
    
    # 2.5. Reconstruction des droits
    rights = []
    for dataright in datarights:
        datarightSplit = dataright.split(',')
        if len(datarightSplit) == 2:
            right = {
                "id": datarightSplit[0],
                "role": datarightSplit[1]
            }
            rights.append(right)
    
    # 3. Envoi de la donnée à NAKALA
    postdata = { # Variable contenant le contenu de la requête
        "status" : status,
        "files" : files,
        "metas" : metas,
        "rights": rights
    }
    content = json.dumps(postdata) # Serialisation du contenu en JSON
    headers = { # Header de la requête
      'Content-Type': 'application/json',
      'X-API-KEY': apiKey,
    }
    response = requests.request('POST', apiUrl + '/datas', headers=headers, data=content) # Appel à l'API
    if ( 201 == response.status_code ): # On obtient un code 201 si tout s'est bien passé
        parsed = json.loads(response.text) # On parse la réponse de l'API
        print('La donnée ' + str(num) + ' a bien été créée : ' + parsed["payload"]["id"] + '\n') # Affichage d'un message de succès
        outputData[0] = parsed["payload"]["id"] # on stocke les informations nécessaire pour le fichier de sortie
        outputData[3] = 'OK'
        outputData[4] = response.text
    else:
        print("Une erreur s'est produite !") # En cas d'erreur, on affiche un message
        outputData[3] = 'ERROR' # on complète les informations nécessaie pour le fichier de sortie
        outputData[4] = response.text
    outputWriter.writerow(outputData) # On écrit les informations dans le fichier de sortie

# 4. fermeture du fichier de sortie
output.close()

CREATION DE LA DONNEE 0 : Martin-pêcheur
Envoi du fichier kingfisher.jpg...
La donnée 0 a bien été créée : 10.34847/nkl.1cbat5rh

CREATION DE LA DONNEE 1 : Chouette
Envoi du fichier owl-1.jpg...
Envoi du fichier owl-2.jpg...
La donnée 1 a bien été créée : 10.34847/nkl.f9f9b2g7

CREATION DE LA DONNEE 2 : Paon
Envoi du fichier peafowl.jpg...
La donnée 2 a bien été créée : 10.34847/nkl.aaea0zc8



## 4-Script final d'import par lot

Voici la version final du script que nous avons mis en place :

In [6]:
import csv, requests, json, time

# Endpoint API & Clef d'API
apiUrl = 'https://apitest.nakala.fr'
apiKey = 'aae99aba-476e-4ff2-2886-0aaf1bfa6fd2'

# 1.1 Lecture du fichier CSV
with open('simple-dataset/dataset.csv', newline='') as f:
    reader = csv.reader(f)
    dataset = list(reader)
dataset.pop(0) # suppression des titres des colonnes

# 1.2 Préparation d'un fichier de sortie
output = open('output.csv', 'w') # ouverture d'un fichier en mode écriture
outputWriter = csv.writer(output) # création d'un objet pour écrire dans ce fichier
header = ['identifier', 'files', 'title', 'status', 'response'] # nom des colonnes à insérer dans ce fichier
outputWriter.writerow(header) # écriture du nom des colonnes dans ce fichier

# 2. Parcours des différentes lignes du fichier
for num, data in enumerate(dataset):
    
    # 2.1. Récupération des infos disponibles sur la donnée à créer
    filenames = data[0].split(';') # on distingue bien les valeurs séparées par des points-virgules
    status = data[1]
    datatype = data[2]
    title = data[3]
    authors = data[4].split(';')
    date = data[5]
    license = data[6]
    description = data[7]
    keywords = data[8].split(';')
    datarights = data[9].split(';')
    
    outputData = ['','',title,'','']; # variable où seront stockées les informations à écrire dans le fichier de sortie
    
    print('CREATION DE LA DONNEE ' + str(num) + " : " + title)
    
    # 2.2. Envoi des fichiers à l'API
    files = [] # variable pour stocker les informations retournées en JSON par l'API à chaque upload.
    outputFiles = [] # variable pour stocker les informations à écrire dans le fichier de sortie concernant les fichiers uploadés
    for filename in filenames: # on parcours l'ensemble des fichiers d'une donnée
        goToNextData = False # variable pour savoir s'il faut passer à la donnée suivante en cas d'erreur lors d'un upload
        print('Envoi du fichier ' + filename + '...') # on affiche un message pour le suivi de l'upload
        # écriture de la requête à l'API (ne contient pas de body en JSON, mais un fichier et un clef d'API)
        payload={}
        postfiles=[('file',(filename,open('./simple-dataset/img/' + filename, 'rb'),'image/jpeg'))]
        headers = {'X-API-KEY': apiKey }
        # appel à l'API pour uploader le fichier
        response = requests.request('POST', apiUrl + '/datas/uploads', headers=headers, data=payload, files=postfiles)
        # si l'upload s'est bien passé, on stocke les informations retournés par l'API dans la variable 'files'
        if ( 201 == response.status_code ):
            # avant de stocker les informations retournées par l'API sur le fichier, on y ajoute une date d'embargo
            file = json.loads(response.text)
            file["embargoed"] = time.strftime("%Y-%m-%d") # on renseigne la date du jour
            files.append(file)
            # ajout du nom du fichier et du sha1 aux informations à écrire dans le fichier de sortie
            outputFiles.append(filename + ',' + file["sha1"])
        else:
            # une erreur s'est produite avec un upload
            outputFiles.append(filename) # on ajoute le nom du fichier aux informations à écrire dans le fichier de sortie
            outputData[1] = ';'.join(outputFiles)
            outputData[3] = 'ERROR' # on complète les informations à écrire dans le fichier de sortie en indiquant qu'il y a eu une erreur
            outputData[4] = response.text # on insère ici le retour de l'API
            outputWriter.writerow(outputData) # on écrit dans le fichier de sortie
            print ("Certains fichiers n'ont pas pu être envoyés, on passe à la donnée suivante...") # on affiche un message d'erreur
            goToNextData = True # on indique dans cette variable qu'il faut passer à la donnée suivante
            break # on stop l'upload des autres fichiers de la donnée
    if goToNextData: continue # on passe à la donnée suivante s'il y a eu une erreur
    
    # 2.3. On garde une trace pour le fichier de sortie de la liste des fichiers uploadés
    outputData[1] = ';'.join(outputFiles)
    
    # 2.4. Reconstruction des métadonnées
    metas = [] # on stocke dans cette variable l'ensemble des métadonnées dans le format attendu
    
    # la métadonnée type (obligatoire)
    metaType = {
        "value": datatype, # on insère ici le contenu de la colonne "datatype" (un seul type par donnnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#anyURI", # on indique ici que la valeur renseignée est de type URI
        "propertyUri": "http://nakala.fr/terms#type" # on indique ici le champ "type" issu du vocabulaire NAKALA
    }
    metas.append(metaType) # ajout de la métadonnée dans le tableau "metas"
    
    # la métadonnée titre (obligatoire)
    metaTitle = {
        "value": title, # on insère ici le contenu de la colonne "title" (un seul titre par donnnée)
        "lang": "fr", # on indique ici la langue du titre (cf. ISO-639-1 ou ISO-639-3 pour les langues moins courantes)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string", # on indique ici que la valeur renseignée est une chaîne de caractères
        "propertyUri": "http://nakala.fr/terms#title" # on indique ici le champ "title" issu du vocabulaire de NAKALA
    }
    metas.append(metaTitle)
    
    # les métadonnées auteurs (obligatoire pour une donnée publiée)
    for author in authors: # la colonne "authors" peut comporter plusieurs valeurs qu'on parcours une à une
        # pour chaque valeur, on sépare le nom et le prénom pour construire la métadonnée nakala:creator
        surnameGivenname = author.split(',')
        metaAuthor = {
            "value": { # la valeur de cette métadonnée n'est pas une simple chaîne de caractères, mais un objet composé d'au minimum deux propriétées (givenname et surname)
                "givenname": surnameGivenname[1], # on insère ici le prénom
                "surname": surnameGivenname[0] # on insère ici le nom de famille
            },
            "propertyUri": "http://nakala.fr/terms#creator" # on indique ici le champ "creator" issu du vocabulaire de NAKALA
        }
        metas.append(metaAuthor)
        
    # la métadonnée date de création (obligatoire pour une donnée publiée)
    metaCreated = {
        "value": date, # on insère ici le contenu de la colonne "date" (une seule date de création par donnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#created"
    }
    metas.append(metaCreated)
    
    # la métadonnée licence (obligatoire pour une donnée publiée)
    metaLicense = {
        "value": license, # On insère ici le contenu de la colonne "license" (une seule license par donnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#license"
    }
    metas.append(metaLicense)
    
    # la métadonnée description (facultative)
    metaDescription = {
        "value": description, # on insère ici le contenu de la colonne "description" (une seule description par donnée)
        "lang": "fr",
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://purl.org/dc/terms/description" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
    }
    metas.append(metaDescription)
    
    # les métadonnées mots-clés (facultatives)
    for keyword in keywords: # on parcours les valeurs de la colonne "keywords"
        metaKeyword = {
            "value": keyword, # on insère ici la valeur d'un mot-clé
            "lang": "fr",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/subject" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaKeyword)
    
    # 2.5. Reconstruction des droits
    rights = [] # on stocke les droits d'une donnée dans un tableau
    for dataright in datarights: # on parcours les valeurs de la colonne "rights"
        datarightSplit = dataright.split(',') # chaque valeur est composée d'un id et d'un rôle séparé par une virgule. On sépare ces valeurs
        if len(datarightSplit) == 2: # on vérifie qu'on a bien deux valeurs uniquement
            right = { # on reconstruit l'objet "right" composé des propriétés "id" et "role"
                "id": datarightSplit[0],
                "role": datarightSplit[1]
            }
            rights.append(right) # on ajout l'objet "right" dans le tableau "rights"
    
    # 3. Envoi de la donnée à NAKALA
    postdata = { # variable contenant le contenu de la requête
        "status" : status,
        "files" : files,
        "metas" : metas,
        "rights": rights
    }
    content = json.dumps(postdata) # serialisation du contenu en JSON
    headers = { # header de la requête
      'Content-Type': 'application/json',
      'X-API-KEY': apiKey,
    }
    response = requests.request('POST', apiUrl + '/datas', headers=headers, data=content) # requête à l'API
    if ( 201 == response.status_code ): # on obtient un code 201 si tout s'est bien passé
        parsed = json.loads(response.text) # on parse la réponse de l'API
        print('La donnée ' + str(num) + ' a bien été créée : ' + parsed["payload"]["id"] + '\n') # affichage d'un message de succès
        outputData[0] = parsed["payload"]["id"] # on stocke les informations nécessaire pour le fichier de sortie
        outputData[3] = 'OK'
        outputData[4] = response.text
    else:
        print("Une erreur s'est produite !") # en cas d'erreur, on affiche un message
        outputData[3] = 'ERROR' # on complète les informations nécessaires pour le fichier de sortie
        outputData[4] = response.text
    outputWriter.writerow(outputData) # on écrit les informations dans le fichier de sortie

# 4. Fermeture du fichier de sortie
output.close()

CREATION DE LA DONNEE 0 : Martin-pêcheur
Envoi du fichier kingfisher.jpg...
La donnée 0 a bien été créée : 10.34847/nkl.b8d3qh01

CREATION DE LA DONNEE 1 : Chouette
Envoi du fichier owl-1.jpg...
Envoi du fichier owl-2.jpg...
La donnée 1 a bien été créée : 10.34847/nkl.789fy3x6

CREATION DE LA DONNEE 2 : Paon
Envoi du fichier peafowl.jpg...
La donnée 2 a bien été créée : 10.34847/nkl.fb3e7zx8



## 5-Visualisation des résultats

Pour visualiser le résultat, vous pouvez par exemple vous connecter à l'interface web de [NAKALA test](https://test.nakala.fr) avec le compte `unakala3` et aller visualiser dans la liste des données du compte que l'import s'est bien déroulé :

![Liste des données importées](./illustrations/resultat-import-1.png)

![Métadonnées d'une des données importées](./illustrations/resultat-import-2.png)

## 6-Pour aller plus loin

### 6.1-Import du jeu de données du module 2 de l'ANF

Vous pouvez prolonger ce TP en reprenant le [jeu de données](./dataset-anf/DonneesOrganisees.zip) présenté lors du [module 2](https://anf2021-humanum.sciencesconf.org/resource/page/id/1) de l'ANF. Au lieu d'effectuer le dépôt de chaque donnée à la main comme nous l'avons fait lors du TP sur le dépôt dans NAKALA via l'interface web, vous pouvez écrire votre propre script afin d'effectuer ces dépôts par lot. Pour cela, vous pouvez vous inspirer du script que nous venons de mettre en place en l'adaptant à ce jeu de données plus complexe. Voici quelques pistes :

- reprenez le [tableau des métadonnées](./dataset-anf/metadata_cleaned.xlsx) et adaptez-le afin qu'il soit plus facilement exploitable par le script que vous allez mettre en place (ex: ajoutez une colonne pour gérer les droits, modifiez certaines valeurs pour coller à celles attendues par l'API ou prévoyez une étape de mapping dans votre script...)
- faites un export CSV de votre nouveau tableau (attention à l'encodage en UTF-8 et au caractère séparateur que vous avez choisi)
- modifier le script d'import par lot pour prendre en compte une date d'embargo, des métadonnées dans plusieurs langues, des auteurs anonymes et des dates inconnues, l'ajout d'autres métadonnées `dcterms:*` ou encore l'ajout de données dans des collections

### 6.2-Proposition de réponse

Voici une proposition de script permettant d'effectuer le dépôt des données du module 2 de l'ANF. Nous partons d'un état où les collections `TGIR Huma-Num`, `NAKALA` et `ISIDORE` sont déjà créées ainsi que les listes utilisateurs `HUMA-NUM (admin)` et `HUMA-NUM (lecteur)`. Créez-les si ce n'est pas déjà le cas. Nous avons adapté le [tableau de métadonnées](./dataset-anf/metadata_cleaned.xlsx) en créant un nouveau [fichier CSV](./dataset-anf/metadata_cleaned_for_import.csv) plus facilement exploitable par notre script. Pour que ce script fonctionne, vous devez compléter les variables suivantes : `apiUrl`, `apiKey`, `nakalaCollectionDict`, `nakalaGroupDict`

In [ ]:
import csv, requests, json, time

# Endpoint & Clef d'API
apiUrl = '' # A COMPLETER ! (ex: https://apitest.nakala.fr)
apiKey = '' # A COMPLETER ! (ex: aab91cae-ef74-bb47-a35d-dc265648be86)

# Mapping CSV vers API (on aurait aussi pu remplacer directement ces valeurs dans le CSV)
nakalaTypeDict = {
    "Article de journal": "http://purl.org/coar/resource_type/c_6501",
    "Cours": "http://purl.org/coar/resource_type/c_e059",
    "Image": "http://purl.org/coar/resource_type/c_c513",
    "Poster": "http://purl.org/coar/resource_type/c_6670",
    "Présentation": "http://purl.org/coar/resource_type/c_c94f",
    "Set de données": "http://purl.org/coar/resource_type/c_ddb1",
    "Texte": "http://purl.org/coar/resource_type/c_18cf"
}
nakalaCollectionDict = {
    "TGIR Huma-Num": "", # A COMPLETER ! (ex: 10.34847/nkl.f5bbj82f)
    "NAKALA": "", # A COMPLETER !
    "ISIDORE": "" # A COMPLETER !
}
nakalaGroupDict = {
    "HUMA-NUM (admin)": "", # A COMPLETER ! (ex: f284379d-1ab3-11ec-9c38-0242ac120003)
    "HUMA-NUM (lecteur)": "" # A COMPLETER !
}

# 1.1 Lecture du fichier CSV
with open('dataset-anf/metadata_cleaned_for_import.csv', newline='') as f:
    reader = csv.reader(f)
    dataset = list(reader)
dataset.pop(0) # suppression des titres des colonnes

# 1.2 Préparation d'un fichier de sortie
output = open('output.csv', 'w') # ouverture d'un fichier en mode écriture
outputWriter = csv.writer(output) # création d'un objet pour écrire dans ce fichier
header = ['identifier', 'files', 'title', 'status', 'response'] # nom des colonnes à insérer dans ce fichier
outputWriter.writerow(header) # écriture du nom des colonnes dans ce fichier

# 2. Parcours des différentes lignes du fichier
for num, data in enumerate(dataset):
    # 2.1. Récupération des infos disponibles sur la donnée à créer
    filenames = data[0].split(';') # on distingue bien les valeurs séparées par des points-virgules
    embargo = data[1]
    datatype = data[2]
    titleFr = data[3]
    titleEs = data[4]
    authors = list(filter(None,data[5].split(';'))) # ajout d'un filtre pour ne pas garder les valeurs vides
    date = data[6]
    license = data[7]
    descriptionFr = data[8]
    descriptionEs = data[9]
    keywordsFr = list(filter(None,data[10].split(';')))
    keywordsEs = list(filter(None,data[11].split(';')))
    keywordsURI = list(filter(None,data[12].split(';')))
    language = data[13]
    dctermsCreator = data[14]
    dctermsCreated = data[15]
    dctermsPublisher = data[16]
    nakalaCollections = list(filter(None,data[17].split(';')))
    datarights = list(filter(None,data[18].split(';')))
    
    outputData = ['','',titleFr,'','']; # variable où seront stockées les informations à écrire dans le fichier de sortie
    
    print('CREATION DE LA DONNEE ' + str(num) + " : " + titleFr)
    
    # 2.2. Envoi des fichiers à l'API
    files = [] # variable pour stocker les informations retournées en JSON par l'API à chaque upload.
    outputFiles = [] # variable pour stocker les informations à écrire dans le fichier de sortie concernant les fichiers uploadés
    for filename in filenames: # on parcours l'ensemble des fichiers d'une donnée
        print('Envoi du fichier ' + filename + '...') # on affiche un message pour le suivi de l'upload
        goToNextData = False
        # écriture de la requête à l'API (ne contient pas de body en JSON, mais un fichier et un clef d'API)
        payload={}
        postfiles=[('file',(filename,open('./dataset-anf/files/' + filename, 'rb')))]
        headers = {'X-API-KEY': apiKey }
        # appel à l'API pour uploader le fichier
        response = requests.request("POST", apiUrl + '/datas/uploads', headers=headers, data=payload, files=postfiles)
        # si l'upload s'est bien passé, on stocke les informations retournés par l'API dans la variable 'files'
        if ( 201 == response.status_code ):
            # avant de stocker les informations retournées par l'API sur le fichier, on y ajoute une date d'embargo
            file = json.loads(response.text)
            file["embargoed"] = embargo or time.strftime("%Y-%m-%d") # on renseigne la date du jour si pas de date d'embargo renseignée
            files.append(file)
            # ajout du nom du fichier et du sha1 aux informations à écrire dans le fichier de sortie
            outputFiles.append(filename + ',' + file["sha1"])
        else:
            # une erreur s'est produite avec un upload
            outputFiles.append(filename) # on ajoute le nom du fichier aux informations à écrire dans le fichier de sortie
            outputData[1] = ';'.join(outputFiles)
            outputData[3] = 'ERROR' # on complète les informations à écrire dans le fichier de sortie en indiquant qu'il y a eu une erreur
            outputData[4] = response.text # on insère ici le retour de l'API
            outputWriter.writerow(outputData) # on écrit dans le fichier de sortie
            print ("Certains fichiers n'ont pas pu être envoyés, on passe à la donnée suivante...") # on affiche un message d'erreur
            goToNextData = True # on stocke dans cette variable qu'il y a eu un problème
            break # on arrête l'upload des autres fichiers de la donnée
    if goToNextData: continue # on passe à la donnée suivante si
    
    # 2.3. On garde une trace pour le fichier de sortie de la liste des fichiers uploadés
    outputData[1] = ';'.join(outputFiles)
    
    # 2.4. Reconstruction des métadonnées
    metas = [] # on stocke dans cette variable l'ensemble des métadonnées dans le format attendu
    
    # la métadonnée type (obligatoire)
    metaType = {
        "value": nakalaTypeDict[datatype], # on insère ici le contenu de la colonne "datatype" mappé sur l'URI correspondante (un seul type par donnnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#anyURI", # on indique ici que la valeur renseignée est de type URI
        "propertyUri": "http://nakala.fr/terms#type" # on indique ici le champ "type" issu du vocabulaire NAKALA
    }
    metas.append(metaType) # ajout de la métadonnée dans le tableau "metas"
    
    # la métadonnée titre (obligatoire)
    metaTitleFr = {
        "value": titleFr, # on insère ici le contenu de la colonne "nakala:title (fr)" (un seul titre fr par donnnée)
        "lang": "fr", # on indique ici la langue du titre (cf. ISO-639-1 ou ISO-639-3 pour les langues moins courantes)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string", # on indique ici que la valeur renseignée est une chaîne de caractères
        "propertyUri": "http://nakala.fr/terms#title" # on indique ici le champ "title" issu du vocabulaire de NAKALA
    }
    metas.append(metaTitleFr)
    # ajout d'un titre en espagnol (si présent)
    if titleEs:
        metaTitleEs = {
            "value": titleEs, # on insère ici le contenu de la colonne "nakala:title (es)" (un seul titre es par donnnée)
            "lang": "es", # on indique ici la bonne langue
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://nakala.fr/terms#title"
           }
        metas.append(metaTitleEs)
    
    # les métadonnées auteurs (obligatoire pour une donnée publiée)
    if not authors: # si vide, on ajoute une métadonnée nakala:creator "anonyme"
       metaAuthor = {
          "value": None, # sic. None sans guillemet veut dire "null" en Python
          "propertyUri": "http://nakala.fr/terms#creator" # on indique ici le champ "creator" issu du vocabulaire de NAKALA
       }
       metas.append(metaAuthor)
    else:    
        for author in authors: # la colonne "nakala:creator" peut comporter plusieurs valeurs qu'on parcours une à une
               # pour chaque valeur, on sépare le nom et le prénom pour construire la métadonnée nakala:creator
            surnameGivennameORCID = author.split(',')
            metaAuthor = {
                "value": { # la valeur de cette métadonnée n'est pas une simple chaîne de caractères, mais un objet composé d'au minimum deux propriétées (givenname et surname)
                    "givenname": surnameGivennameORCID[1], # on insère ici le prénom
                    "surname": surnameGivennameORCID[0] # on insère ici le nom de famille
                },
                "propertyUri": "http://nakala.fr/terms#creator" # on indique ici le champ "creator" issu du vocabulaire de NAKALA
            }
            # on ajoute le numéro ORCID (si présent)
            if len(surnameGivennameORCID) == 3:
                metaAuthor["value"]["orcid"] = surnameGivennameORCID[2] # on insère ici le numéro ORCID
            # ajout de la métadonnée
            metas.append(metaAuthor)
       
    # la métadonnée date de création (obligatoire pour une donnée publiée)
    if not date: # si vide, on ajoute une métadonnée "nakala:created" "inconnue"
        metaCreated = {
            "value": None,
            "propertyUri": "http://nakala.fr/terms#created"
        }
    else:
        metaCreated = {
            "value": date, # on insère ici le contenu de la colonne "nakala:created" (une seule date de création par donnée)
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://nakala.fr/terms#created"
        }
    metas.append(metaCreated)
    
    # la métadonnée licence (obligatoire pour une donnée publiée)
    metaLicense = {
        "value": license, # On insère ici le contenu de la colonne "nakala:license" (une seule licence par donnée)
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://nakala.fr/terms#license"
    }
    metas.append(metaLicense)
    
    # la métadonnée description (facultative)
    metaDescriptionFr = {
        "value": descriptionFr, # on insère ici le contenu de la colonne "dcterms:description (fr)" (une seule description par donnée)
        "lang": "fr",
        "typeUri": "http://www.w3.org/2001/XMLSchema#string",
        "propertyUri": "http://purl.org/dc/terms/description" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
    }
    metas.append(metaDescriptionFr)
    # ajout de la description en espagnol (si présent)
    if descriptionEs:
        metaDescriptionEs = {
            "value": descriptionEs,
            "lang": "es",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/description"
        }
        metas.append(metaDescriptionEs)
    
    # les métadonnées mots-clés (facultatives)
    for keywordFr in keywordsFr: # on parcours les valeurs de la colonne "dcterms:subject (fr)"
        metaKeywordFr = {
            "value": keywordFr, # on insère ici la valeur d'un mot-clé
            "lang": "fr",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/subject" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaKeywordFr)
    # mots-clés en espagnol (si présent)
    for keywordEs in keywordsEs: # on parcours les valeurs de la colonne "dcterms:subject (es)"
        metaKeywordEs = {
            "value": keywordEs, # on insère ici la valeur d'un mot-clé
            "lang": "es",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/subject"
        }
        metas.append(metaKeywordEs)
    # mots-clés de type URI (si présent)
    for keywordURI in keywordsURI: # on parcours les valeurs de la colonne "dcterms:subject (URI)"
        metaKeywordURI = {
            "value": keywordURI, # on insère ici la valeur d'un mot-clé
            "typeUri": "http://www.w3.org/2001/XMLSchema#anyURI", # on indique ici qu'il s'agit bien d'une URI
            "propertyUri": "http://purl.org/dc/terms/subject"
        }
        metas.append(metaKeywordURI)
    
    # la métadonnée de "dcterms:language" (facultative)
    if language:
        metaLanguage = {
            "value": language,
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/language" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaLanguage)
    
    # la métadonnée "dcterms:creator" (pour les formes auteurs qui ne distingue pas nom,prénom)
    if dctermsCreator:
        metaDctermsCreator = {
            "value": dctermsCreator, # on insère ici le contenu de la colonne "dcterms:creator"
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/creator" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaDctermsCreator)
        
    # la métadonnée "dcterms:created" (pour les dates qui ne respectent pas le format AAAA, AAAA-MM, AAAA-MM-JJ)
    if dctermsCreated:
        metaDctermsCreated = {
            "value": dctermsCreated, # on insère ici le contenu de la colonne "dcterms:created" (une seule date par donnée)
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/created" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaDctermsCreated)
        
    # la métadonnée "dcterms:publisher"
    if dctermsPublisher:
        metaDctermsPublisher = {
            "value": dctermsPublisher, # on insère ici le contenu de la colonne "dcterms:publisher" (un seul publisher par donnée)
            "lang": "fr",
            "typeUri": "http://www.w3.org/2001/XMLSchema#string",
            "propertyUri": "http://purl.org/dc/terms/publisher" # notez qu'il ne s'agit plus ici d'une propriété issue du vocabulaire NAKALA, mais du vocabulaire Dcterms
        }
        metas.append(metaDctermsPublisher)
     
    # 2.5. Reconstruction des droits
    rights = [] # on stocke les droits d'une donnée dans un tableau
    for dataright in datarights: # on parcours les valeurs de la colonne "rights"
        datarightSplit = dataright.split(',') # chaque valeur est composée d'un id et d'un rôle séparé par une virgule. On sépare ces valeurs
        if len(datarightSplit) == 2: # on vérifie qu'on a bien deux valeurs uniquement
            right = { # on reconstruit l'objet "right" composé des propriétés "id" et "role"
                "id": nakalaGroupDict[datarightSplit[0]], #on mappe la valeur du CSV avec l'id
                "role": datarightSplit[1]
            }
            rights.append(right) # on ajout l'objet "right" dans le tableau "rights"
    
    # 2.6. Reconstruction des collections
    collectionsIds = []
    for nakalaCollection in nakalaCollections:
        collectionsIds.append(nakalaCollectionDict[nakalaCollection])
    
    # 3. Envoi de la donnée à NAKALA
    postdata = { # variable contenant le contenu de la requête
        "status" : "published", # on publie directement les données
        "files" : files,
        "metas" : metas,
        "rights": rights,
        "collectionsIds": collectionsIds
    }
    content = json.dumps(postdata) # serialisation du contenu en JSON
    headers = { # header de la requête
      'Content-Type': 'application/json',
      'X-API-KEY': apiKey,
    }
    response = requests.request("POST", apiUrl + '/datas', headers=headers, data=content) # requête à l'API
    if ( 201 == response.status_code ): # on obtient un code 201 si tout s'est bien passé
        parsed = json.loads(response.text) # on parse la réponse de l'API
        print('La donnée ' + str(num) + ' a bien été créée : ' + parsed["payload"]["id"] + '\n') # affichage d'un message de succès
        outputData[0] = parsed["payload"]["id"] # on stocke les informations nécessaire pour le fichier de sortie
        outputData[3] = 'OK'
        outputData[4] = response.text
    else:
        print("Une erreur s'est produite !") # en cas d'erreur, on affiche un message
        outputData[3] = 'ERROR' # on complète les informations nécessaires pour le fichier de sortie
        outputData[4] = response.text
    outputWriter.writerow(outputData) # on écrit les informations dans le fichier de sortie

# 4. Fermeture du fichier de sortie
output.close()

## 7-Conclusion

Nous espérons que ce TP vous aura permi de comprendre plus concrètement comment fonctionne l'API de NAKALA et ce qu'elle permet de mettre en place.

Signalons quelques projets simplifiant l'import par lot dans NAKALA :

- [MyNakala](http://mynakala.huma-num.fr/) : Il s'agit d'une application développée par le [Consortium CAHIER](https://cahier.hypotheses.org/) et qui vous permet de déposer facilement vos données sur Nakala. MyNakala utilise l'API de Nakala pour réaliser le dépôt de grandes quantités de données.

Pour toute question ou remarque sur ce notebook, n'hésitez pas à nous écrire à nakala@huma-num.fr